# Sentiment Analysis

Adds `Sentiment_Label` and `Sentiment_Score#number` columns to your survey by scoring a text column with a RoBERTa sentiment model via the HuggingFace Inference API.

A free HuggingFace token improves rate limits but is not required. Get one at https://huggingface.co/settings/tokens

In [ ]:
# ── Colab bootstrap (no-op on Binder / JupyterHub) ────────────────────────
import os, sys
if "google.colab" in sys.modules:
    if not os.path.isfile("/tmp/suave-nb/helpers/suave_utils.py"):
        import subprocess
        _r = subprocess.run(
            ["git", "clone", "--depth=1", "https://github.com/izaslavsky/suave-notebooks.git", "/tmp/suave-nb"],
            capture_output=True, text=True
        )
        if _r.returncode != 0:
            raise RuntimeError(f"Could not clone suave-notebooks:\n{_r.stderr}")
    sys.path.insert(0, "/tmp/suave-nb/helpers")


In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

# Colab only: paste values from the SuAVE launch dialog
if not su.ENV.colab:
    # Binder / JupyterHub: parameters load automatically
    _p = su.load_params()
    if _p is None:
        raise RuntimeError('No SuAVE params. Open via SuAVEDispatch.ipynb first.')
else:
    # Colab: use Drive (automatic) or enter credentials below
    SUAVE_TOKEN = ''   # @param {type:"string"}
    SUAVE_HOST  = ''   # @param {type:"string"}
    _p = su.load_params(token=SUAVE_TOKEN, host=SUAVE_HOST)

user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'
_prefix       = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '/')
full_notebook_url = _prefix.rstrip('/') + '/lab/tree/SuAVEDispatch.ipynb'

localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images

absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)

In [ ]:
import sys
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint

import ipywidgets as widgets
import pandas as pd
import numpy as np

In [ ]:
# Optional HuggingFace token — improves rate limits, not required
# Get a free token at https://huggingface.co/settings/tokens
HF_TOKEN = ''   # @param {type:"string"}

client = su.get_hf_client(HF_TOKEN)

## 1. Load survey data and select text column

In [ ]:
df = panellibs.extract_data(absolutePath + csv_file)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")

# Text columns: plain string columns and #long columns
text_cols = [c for c in df.columns if '#number' not in c and '#img' not in c
             and '#hidden' not in c and '#date' not in c]

col_selector = widgets.Dropdown(
    options=text_cols,
    description='Text column:',
    layout=widgets.Layout(width='60%')
)
display(col_selector)

## 2. Run sentiment analysis

In [ ]:
MODEL = 'cardiffnlp/twitter-roberta-base-sentiment-latest'

def _score(text):
    if not text or pd.isna(text) or str(text).strip() == '':
        return None, None
    try:
        result = client.text_classification(str(text)[:512], model=MODEL)
        top = result[0]
        label = top.label.split('_')[-1].capitalize()  # LABEL_0 → Negative etc
        score = round(top.score, 4)
        return label, score
    except Exception:
        # Fallback: local transformers pipeline if HF API unavailable
        return None, None

col = col_selector.value
printmd(f"**Scoring `{col}` with `{MODEL}`...**")

labels, scores = [], []
from tqdm.auto import tqdm

# Try HF Inference API first; fall back to local transformers pipeline
api_ok = True
try:
    test_result = client.text_classification('test', model=MODEL)
except Exception as e:
    api_ok = False
    printmd(f"**HF API unavailable ({e}). Falling back to local pipeline.**")

if api_ok:
    for text in tqdm(df[col], desc='Sentiment'):
        lbl, sc = _score(text)
        labels.append(lbl)
        scores.append(sc)
else:
    try:
        from transformers import pipeline
        pipe = pipeline('sentiment-analysis', model=MODEL, truncation=True, max_length=512)
        for text in tqdm(df[col], desc='Sentiment (local)'):
            if not text or pd.isna(text):
                labels.append(None); scores.append(None)
            else:
                r = pipe(str(text)[:512])[0]
                labels.append(r['label'].split('_')[-1].capitalize())
                scores.append(round(r['score'], 4))
    except ImportError:
        raise RuntimeError('Install transformers: !pip install transformers torch')

df['Sentiment_Label'] = labels
df['Sentiment_Score#number'] = scores

printmd('**Done. Sample results:**')
display(df[[col, 'Sentiment_Label', 'Sentiment_Score#number']].dropna().head(10))

## 3. Review distribution

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['Sentiment_Label'].value_counts().plot.bar(ax=axes[0], color=['#e74c3c','#95a5a6','#2ecc71'])
axes[0].set_title('Label distribution')
axes[0].set_xlabel('')

df['Sentiment_Score#number'].dropna().plot.hist(ax=axes[1], bins=30, color='#3498db', edgecolor='white')
axes[1].set_title('Score distribution')
plt.tight_layout()
plt.show()

## 4. Save results and publish to SuAVE

In [ ]:
new_file = suaveint.save_csv_file(df, absolutePath, csv_file)

In [ ]:
input_text = widgets.Text(placeholder='Enter survey name...')
output_text = widgets.Text()

def _bind(sender):
    output_text.value = input_text.value

input_text.on_submit(_bind)
printmd("**Enter survey name, press Enter, then run the next cell:**")
display(input_text, output_text)

In [ ]:
survey_name = output_text.value
suaveint.create_survey(survey_url, new_file, survey_name, dzc_file, user, csv_file, view, views)